In [1]:
# =========================================================================
# GRAZIOSO SALVARE — PROJECT TWO DASHBOARD  (v5 — static layout auth gate)
# Full Stack MongoDB Dashboard
# Student: Zion Kiniebrew-Jenkins
#
# KEY DESIGN: every component lives in the static layout from app start.
# Login screen / dashboard are toggled via display:none/block in callbacks.
# This guarantees Dash can always wire callbacks to IDs regardless of
# which "page" the user is on.
# =========================================================================

from jupyter_dash import JupyterDash
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
from dash import callback_context
import dash_leaflet as dl
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import base64
import os
from datetime import datetime

from CRUD_Python_Module import AnimalShelter

# =========================================================================
# 1. LOGO
# =========================================================================
encoded_image = ''
try:
    encoded_image = base64.b64encode(
        open('Grazioso Salvare Logo.png', 'rb').read()
    ).decode()
except FileNotFoundError:
    print("Logo not found — continuing without it.")

logo_src = f'data:image/png;base64,{encoded_image}' if encoded_image else None

# =========================================================================
# 2. THEME TOKENS
# =========================================================================
THEMES = {
    "dark": {
        "bg":          "#0d1117",
        "surface":     "#161b22",
        "surface2":    "#1c2128",
        "border":      "#30363d",
        "text":        "#e6edf3",
        "text_muted":  "#8b949e",
        "accent":      "#58a6ff",
        "accent2":     "#3fb950",
        "accent3":     "#f78166",
        "accent4":     "#d2a8ff",
        "row_odd":     "#1c2128",
        "row_even":    "#161b22",
        "shadow":      "0 4px 24px rgba(0,0,0,0.55)",
        "map_tile":    "https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png",
        "plotly_tmpl": "plotly_dark",
    },
    "light": {
        "bg":          "#f0f4f8",
        "surface":     "#ffffff",
        "surface2":    "#f6f8fa",
        "border":      "#d0d7de",
        "text":        "#1c2128",
        "text_muted":  "#57606a",
        "accent":      "#0969da",
        "accent2":     "#1a7f37",
        "accent3":     "#cf222e",
        "accent4":     "#8250df",
        "row_odd":     "#f6f8fa",
        "row_even":    "#ffffff",
        "shadow":      "0 2px 12px rgba(0,0,0,0.10)",
        "map_tile":    "https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
        "plotly_tmpl": "plotly_white",
    },
}

# =========================================================================
# 3. CSS
# =========================================================================
CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@400;600;800&display=swap');

:root {
  --bg: #0d1117; --surface: #161b22; --surface2: #1c2128;
  --border: #30363d; --text: #e6edf3; --muted: #8b949e;
  --accent: #58a6ff; --accent2: #3fb950; --accent3: #f78166; --accent4: #d2a8ff;
  --shadow: 0 4px 24px rgba(0,0,0,0.55); --radius: 12px;
}
[data-theme="light"] {
  --bg: #f0f4f8; --surface: #ffffff; --surface2: #f6f8fa;
  --border: #d0d7de; --text: #1c2128; --muted: #57606a;
  --accent: #0969da; --accent2: #1a7f37; --accent3: #cf222e; --accent4: #8250df;
  --shadow: 0 2px 12px rgba(0,0,0,0.10);
}
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: "Syne", sans-serif; background: var(--bg); color: var(--text); transition: background .3s, color .3s; }

/* Stripe */
.accent-stripe { height: 3px; background: linear-gradient(90deg, var(--accent), var(--accent2), var(--accent3), var(--accent4), var(--accent)); background-size: 400% 100%; animation: stripe 5s linear infinite; }
@keyframes stripe { 0%{background-position:0%} 100%{background-position:400%} }

/* ── LOGIN ── */
.login-screen { min-height: 100vh; display: flex; align-items: center; justify-content: center; background: var(--bg); }
.login-card { background: var(--surface); border: 1px solid var(--border); border-radius: 18px; padding: 44px 48px; width: 100%; max-width: 420px; box-shadow: var(--shadow); display: flex; flex-direction: column; align-items: center; gap: 6px; }
.login-logo { height: 64px; width: auto; border-radius: 10px; margin-bottom: 6px; }
.login-logo-ph { width: 64px; height: 64px; border-radius: 10px; background: var(--accent); color: #fff; display: flex; align-items: center; justify-content: center; font-size: 1.6rem; font-weight: 800; margin-bottom: 6px; }
.login-title { font-size: 1.45rem; font-weight: 800; letter-spacing: -0.02em; color: var(--text); text-align: center; }
.login-sub { font-size: 0.73rem; letter-spacing: 0.06em; color: var(--muted); text-align: center; margin-bottom: 16px; }
.login-sub0 { font-size: 1.00rem; letter-spacing: 0.06em; color: var(--muted); font-weight: 800; text-align: center; margin-bottom: 18px; }
.login-divider { width: 100%; height: 1px; background: var(--border); margin: 4px 0 18px; }
.lbl { width: 100%; font-size: 0.68rem; letter-spacing: 0.14em; text-transform: uppercase; font-weight: 700; color: var(--muted); margin-bottom: 5px; }
.login-input { width: 100%; padding: 10px 14px; background: var(--surface2); border: 1px solid var(--border); border-radius: 8px; color: var(--text); font-family: "JetBrains Mono", monospace; font-size: 0.9rem; outline: none; transition: all .2s; margin-bottom: 12px; }
.login-input:focus { border-color: var(--accent); box-shadow: 0 0 0 3px rgba(88,166,255,.14); }
.login-input::placeholder { color: var(--muted); }
.login-btn { width: 100%; padding: 11px; background: var(--accent); color: #fff; border: none; border-radius: 8px; font-family: "Syne", sans-serif; font-size: 0.95rem; font-weight: 700; cursor: pointer; letter-spacing: 0.03em; transition: opacity .2s, transform .1s; margin-top: 4px; }
.login-btn:hover { opacity: .88; }
.login-btn:active { transform: scale(.98); }
.msg-error { width: 100%; padding: 10px 14px; background: rgba(247,129,102,.12); border: 1px solid var(--accent3); border-radius: 8px; color: var(--accent3); font-size: .82rem; text-align: center; margin-top: 8px; }
.msg-ok { width: 100%; padding: 10px 14px; background: rgba(63,185,80,.12); border: 1px solid var(--accent2); border-radius: 8px; color: var(--accent2); font-size: .82rem; text-align: center; margin-top: 8px; }
.login-footer { font-size: .68rem; color: var(--muted); margin-top: 18px; text-align: center; }

/* ── DASHBOARD ── */
.dash-header { display: flex; align-items: center; justify-content: space-between; padding: 14px 32px; background: var(--surface); border-bottom: 1px solid var(--border); position: sticky; top: 0; z-index: 100; transition: background .3s, border-color .3s; }
.dash-title { font-size: 1.5rem; font-weight: 800; letter-spacing: -0.02em; color: var(--text); }
.dash-sub { font-size: .73rem; letter-spacing: .07em; color: var(--muted); margin-top: 2px; }
.dash-logo { height: 50px; width: auto; border-radius: 8px; }
.hbtn { padding: 7px 16px; border-radius: 8px; border: 1px solid var(--border); cursor: pointer; font-family: "Syne", sans-serif; font-size: .82rem; font-weight: 600; color: var(--text); background: var(--surface2); transition: all .2s; white-space: nowrap; }
.hbtn:hover { background: var(--accent); color: #fff; border-color: var(--accent); }
.hbtn-accent { background: var(--accent); color: #fff !important; border-color: var(--accent); }
.hbtn-accent:hover { opacity: .84; }
.hbtn-danger { color: var(--accent3) !important; border-color: var(--accent3); }
.hbtn-danger:hover { background: var(--accent3); color: #fff !important; }
.user-badge { display: flex; align-items: center; gap: 7px; padding: 5px 12px; border-radius: 20px; background: rgba(88,166,255,.1); border: 1px solid var(--accent); font-size: .75rem; font-family: "JetBrains Mono", monospace; color: var(--accent); }
.main { padding: 24px 32px; max-width: 1700px; margin: 0 auto; }

/* KPI */
.kpi-grid { display: grid; grid-template-columns: repeat(5,1fr); gap: 14px; margin-bottom: 22px; }
.kpi-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 18px 20px; position: relative; overflow: hidden; transition: transform .2s, box-shadow .2s, background .3s, border-color .3s; }
.kpi-card::after { content:''; position: absolute; bottom: 0; left: 0; right: 0; height: 2px; background: var(--kpi-color, var(--accent)); transition: height .2s; }
.kpi-card:hover { transform: translateY(-4px); box-shadow: 0 8px 28px rgba(0,0,0,.3); }
.kpi-card:hover::after { height: 4px; }
.kpi-icon { font-size: 1.2rem; margin-bottom: 8px; }
.kpi-val { font-family: "JetBrains Mono", monospace; font-size: 1.85rem; font-weight: 600; }
.kpi-lbl { font-size: .68rem; letter-spacing: .14em; text-transform: uppercase; color: var(--muted); margin-top: 4px; }

/* Filter panel */
.filter-panel { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 18px 24px; margin-bottom: 20px; transition: background .3s, border-color .3s; }
.panel-ttl { font-size: .67rem; letter-spacing: .18em; text-transform: uppercase; font-weight: 700; color: var(--muted); margin-bottom: 14px; }
.filter-row { display: flex; align-items: flex-start; gap: 32px; flex-wrap: wrap; }
.fg { display: flex; flex-direction: column; gap: 7px; }
.fg-lbl { font-size: .68rem; letter-spacing: .12em; text-transform: uppercase; color: var(--muted); }
#name-search { background: var(--surface2); border: 1px solid var(--border); border-radius: 7px; padding: 7px 13px; color: var(--text); font-family: "JetBrains Mono", monospace; font-size: .85rem; width: 200px; outline: none; transition: all .2s; }
#name-search:focus { border-color: var(--accent); box-shadow: 0 0 0 3px rgba(88,166,255,.14); }
#name-search::placeholder { color: var(--muted); }

/* Table */
.tbl-wrap { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; margin-bottom: 22px; transition: background .3s, border-color .3s; }
.tbl-bar { display: flex; align-items: center; justify-content: space-between; padding: 12px 18px; border-bottom: 1px solid var(--border); }
.tbl-ttl { font-weight: 700; font-size: .88rem; color: var(--text); }
#rec-count { font-size: .75rem; font-family: "JetBrains Mono", monospace; color: var(--muted); }

/* Charts */
.charts-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 18px; margin-bottom: 20px; }
.chart-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; transition: background .3s, border-color .3s; }
.chart-bar { display: flex; align-items: center; justify-content: space-between; padding: 11px 16px; border-bottom: 1px solid var(--border); }
.chart-ttl { font-size: .8rem; font-weight: 700; color: var(--text); }
.tab-row { display: flex; gap: 5px; }
.tab-btn { padding: 3px 11px; border-radius: 6px; border: 1px solid var(--border); font-size: .72rem; font-weight: 600; cursor: pointer; font-family: "Syne", sans-serif; background: transparent; color: var(--muted); transition: all .2s; }
.tab-btn:hover, .tab-btn.active { background: var(--accent); color: #fff; border-color: var(--accent); }

/* Map */
.map-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; margin-bottom: 20px; transition: background .3s, border-color .3s; }
.map-bar { display: flex; align-items: center; justify-content: space-between; padding: 11px 16px; border-bottom: 1px solid var(--border); }

/* Status */
.status-bar { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius); padding: 10px 18px; display: flex; align-items: center; gap: 8px; flex-wrap: wrap; font-size: .73rem; font-family: "JetBrains Mono", monospace; color: var(--muted); transition: background .3s, border-color .3s; }

/* Dash overrides */
.dash-radioitems label, .dash-checklist label { color: var(--text) !important; cursor: pointer; font-size: .86rem; }
.dash-radioitems input, .dash-checklist input { cursor: pointer; accent-color: var(--accent); }
.rc-slider-track { background-color: var(--accent) !important; }
.rc-slider-handle { border-color: var(--accent) !important; background: var(--accent) !important; }
.rc-slider-rail { background-color: var(--border) !important; }
.rc-slider-mark-text { color: var(--muted) !important; font-size: .7rem !important; }
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--accent); }
"""

# =========================================================================
# 4. APP
# =========================================================================
app = JupyterDash(__name__, suppress_callback_exceptions=True)

app.index_string = f"""<!DOCTYPE html>
<html>
<head>
  {{%metas%}}
  <title>Grazioso Salvare Dashboard</title>
  {{%favicon%}}
  {{%css%}}
  <style>{CSS}</style>
</head>
<body>{{%app_entry%}}<footer>{{%config%}}{{%scripts%}}{{%renderer%}}</footer></body>
</html>"""

# =========================================================================
# 5. STATIC LAYOUT  — everything in DOM from the start
# =========================================================================
_logo_login = (
    html.Img(src=logo_src, className='login-logo') if logo_src
    else html.Div("GS", className='login-logo-ph')
)
_logo_dash = (
    html.Img(src=logo_src, className='dash-logo') if logo_src
    else html.Div("GS", style={'width':'50px','height':'50px','borderRadius':'8px',
                               'background':'var(--accent)','color':'#fff',
                               'display':'flex','alignItems':'center','justifyContent':'center',
                               'fontWeight':'800','fontSize':'1.3rem'})
)

app.layout = html.Div(
    id='root',
    **{'data-theme': 'dark'},
    style={'minHeight': '100vh', 'background': 'var(--bg)', 'transition': 'background .3s'},
    children=[

        # ── Stores ─────────────────────────────────────────────────────
        dcc.Store(id='auth-store',       data=None),   # None until login succeeds
        dcc.Store(id='theme-store',      data='dark'),
        dcc.Store(id='filtered-store',   data=[]),
        dcc.Store(id='chart-type-store', data='pie'),

        # ── Utilities ──────────────────────────────────────────────────
        dcc.Download(id='download-csv'),
        dcc.Interval(id='clock-tick', interval=60_000, n_intervals=0),

        # ══════════════════════════════════════════════════════════════
        # LOGIN PAGE  (visible by default; hidden after auth)
        # ══════════════════════════════════════════════════════════════
        html.Div(id='login-page', children=[
            html.Div(className='accent-stripe'),
            html.Div(className='login-screen', children=[
                html.Div(className='login-card', children=[
                    _logo_login,
                    html.Div("Grazioso Salvare", className='login-title'),
                    html.Div("A•R•I•D", className='login-sub0'),
                    html.Div("Animal Rescue Intelligence Dashboard", className='login-sub'),
                    html.Div(className='login-divider'),

                    html.Div("Username", className='lbl'),
                    dcc.Input(
                        id='username-input', type='text',
                        value='aacuser', placeholder='Enter username',
                        className='login-input',
                        debounce=False,
                        n_submit=0,
                    ),

                    html.Div("Password", className='lbl'),
                    dcc.Input(
                        id='password-input', type='password',
                        value='', placeholder='Enter password',
                        className='login-input',
                        debounce=False,
                        n_submit=0,
                    ),

                    html.Button(
                        "🔐  Sign In ",
                        id='login-btn', n_clicks=0,
                        className='login-btn',
                    ),

                    # Feedback area — populated by callback
                    html.Div(id='login-feedback'),

                    html.Div(
                        " © 2026 Zion Kiniebrew-Jenkins · Global Rain / SNHU",
                        className='login-footer',
                    ),
                ]),
            ]),
        ]),

        # ══════════════════════════════════════════════════════════════
        # DASHBOARD PAGE  (hidden by default; shown after auth)
        # ══════════════════════════════════════════════════════════════
        html.Div(id='dashboard-page', style={'display': 'none'}, children=[

            # Animated stripe
            html.Div(className='accent-stripe'),

            # ── Header ──────────────────────────────────────────────
            html.Div(className='dash-header', children=[

                html.Div(style={'display':'flex','alignItems':'center','gap':'16px'}, children=[
                    _logo_dash,
                    html.Div([
                        html.Div("Grazioso Salvare", className='dash-title'),
                        html.Div(
                            "A•R•I•D Animal Rescue Intelligence Dashboard  ·  © 2026 Zion Kiniebrew-Jenkinss",
                            className='dash-sub'),
                    ]),
                ]),

                html.Div(style={'display':'flex','alignItems':'center','gap':'10px'}, children=[
                    html.Div(id='live-clock', style={
                        'fontFamily':'"JetBrains Mono",monospace',
                        'fontSize':'.75rem','color':'var(--muted)','marginRight':'6px',
                    }),
                    html.Div(id='user-badge', className='user-badge'),
                    html.Button("☀ Light Mode", id='theme-btn',   n_clicks=0, className='hbtn'),
                    html.Button("⬇ Export CSV", id='export-btn',  n_clicks=0, className='hbtn hbtn-accent'),
                    html.Button("⚙ Filters",   id='panel-btn',    n_clicks=0, className='hbtn'),
                    html.Button("⏻ Logout",    id='logout-btn',   n_clicks=0, className='hbtn hbtn-danger'),
                ]),
            ]),

            # ── Main ────────────────────────────────────────────────
            html.Div(className='main', children=[

                # KPI row
                html.Div(id='kpi-row', className='kpi-grid'),

                # Filter panel
                html.Div(id='filter-panel-wrap', children=[
                    html.Div(className='filter-panel', children=[
                        html.Div("Filters", className='panel-ttl'),
                        html.Div(className='filter-row', children=[

                            html.Div(className='fg', children=[
                                html.Div("Rescue Type", className='fg-lbl'),
                                dcc.RadioItems(
                                    id='filter-type',
                                    options=[
                                        {'label': '⟳  All Animals',            'value': 'reset'},
                                        {'label': '🌊  Water Rescue',          'value': 'water'},
                                        {'label': '⛰  Mountain / Wilderness', 'value': 'mountain'},
                                        {'label': '🔍  Disaster / Tracking',   'value': 'disaster'},
                                    ],
                                    value='reset',
                                    labelStyle={'display':'flex','alignItems':'center',
                                                'marginBottom':'7px','gap':'6px'},
                                ),
                            ]),

                            html.Div(className='fg', style={'flex':'1','minWidth':'240px'}, children=[
                                html.Div("Age Range (weeks)", className='fg-lbl'),
                                dcc.RangeSlider(
                                    id='age-slider',
                                    min=0, max=520, step=4, value=[0, 520],
                                    marks={0:'0',52:'1yr',156:'3yr',
                                           260:'5yr',364:'7yr',520:'10yr'},
                                    tooltip={"placement":"bottom","always_visible":False},
                                ),
                            ]),

                            html.Div(className='fg', children=[
                                html.Div("Sex / Reproductive Status", className='fg-lbl'),
                                dcc.Checklist(
                                    id='sex-filter',
                                    options=[
                                        {'label':'♀  Intact Female',  'value':'Intact Female'},
                                        {'label':'♂  Intact Male',    'value':'Intact Male'},
                                        {'label':'♀  Spayed Female',  'value':'Spayed Female'},
                                        {'label':'♂  Neutered Male',  'value':'Neutered Male'},
                                    ],
                                    value=['Intact Female','Intact Male',
                                           'Spayed Female','Neutered Male'],
                                    labelStyle={'display':'flex','alignItems':'center',
                                                'marginBottom':'6px','gap':'5px'},
                                ),
                            ]),

                            html.Div(className='fg', children=[
                                html.Div("Animal Type", className='fg-lbl'),
                                dcc.Checklist(
                                    id='type-filter',
                                    options=[
                                        {'label':'🐕  Dog',   'value':'Dog'},
                                        {'label':'🐈  Cat',   'value':'Cat'},
                                        {'label':'🐾  Other', 'value':'Other'},
                                    ],
                                    value=['Dog','Cat','Other'],
                                    labelStyle={'display':'flex','alignItems':'center',
                                                'marginBottom':'6px','gap':'5px'},
                                ),
                                html.Div("Search by Name", className='fg-lbl',
                                         style={'marginTop':'12px'}),
                                dcc.Input(
                                    id='name-search', type='text',
                                    placeholder='e.g. Max, Bella …',
                                    debounce=True, style={'width':'100%'},
                                ),
                            ]),
                        ]),
                    ]),
                ]),

                # Table
                html.Div(className='tbl-wrap', children=[
                    html.Div(className='tbl-bar', children=[
                        html.Span("Animal Records", className='tbl-ttl'),
                        html.Span(id='rec-count'),
                    ]),
                    dash_table.DataTable(
                        id='datatable-id',
                        columns=[], data=[],
                        page_size=12,
                        sort_action="native",
                        filter_action="native",
                        row_selectable="single",
                        selected_rows=[0],
                        page_action="native",
                        style_table={
                            'overflowX': 'auto',
                            'minWidth': '100%',
                        },
                        style_header={},
                        style_cell={
                            'minWidth': '130px',
                            'maxWidth': '220px',
                            'whiteSpace': 'nowrap',
                            'overflow': 'hidden',
                            'textOverflow': 'ellipsis',
                        },
                        style_data={'border': 'none'},
                        style_data_conditional=[],
                        style_filter={},
                        css=[
                            {'selector': '.dash-spreadsheet td, .dash-spreadsheet th',
                             'rule': 'display: table-cell !important;'},
                            {'selector': '.dash-spreadsheet tr',
                             'rule': 'display: table-row !important;'},
                        ],
                    ),
                ]),

                # Charts
                html.Div(className='charts-grid', children=[
                    html.Div(className='chart-card', children=[
                        html.Div(className='chart-bar', children=[
                            html.Span("Breed Breakdown", className='chart-ttl'),
                            html.Div(className='tab-row', children=[
                                html.Button("Pie", id='pie-btn', n_clicks=0,
                                            className='tab-btn active'),
                                html.Button("Bar", id='bar-btn', n_clicks=0,
                                            className='tab-btn'),
                            ]),
                        ]),
                        dcc.Graph(id='breed-chart', config={'displayModeBar': False}),
                    ]),
                    html.Div(className='chart-card', children=[
                        html.Div(className='chart-bar', children=[
                            html.Span("Sex & Reproductive Status", className='chart-ttl'),
                        ]),
                        dcc.Graph(id='sex-chart', config={'displayModeBar': False}),
                    ]),
                ]),

                # Map
                html.Div(className='map-card', children=[
                    html.Div(className='map-bar', children=[
                        html.Span("Geolocation — Selected Animal", className='chart-ttl'),
                        html.Span(id='map-label', style={
                            'fontFamily':'"JetBrains Mono",monospace',
                            'fontSize':'.75rem','color':'var(--muted)',
                        }),
                    ]),
                    html.Div(id='map-id'),
                ]),

                # Status
                html.Div(id='status-bar', className='status-bar'),
            ]),
        ]),
    ]
)


# =========================================================================
# 6. CALLBACKS
# =========================================================================

# ── AUTH: single callback owns login + logout ──────────────────────────────
# Both buttons are always in the DOM so Dash can wire them at startup.
# We use callback_context.triggered to know which was clicked.
@app.callback(
    [Output('auth-store',     'data'),
     Output('login-page',     'style'),
     Output('dashboard-page', 'style'),
     Output('login-feedback', 'children')],
    [Input('login-btn',  'n_clicks'),
     Input('logout-btn', 'n_clicks')],
    [State('username-input', 'value'),
     State('password-input', 'value'),
     State('auth-store',     'data')],
    prevent_initial_call=True,
)
def handle_auth(login_clicks, logout_clicks, username, password, current_auth):
    triggered = callback_context.triggered[0]['prop_id'].split('.')[0]

    SHOW = {'display': 'block'}
    HIDE = {'display': 'none'}

    # ── LOGOUT
    if triggered == 'logout-btn':
        msg = html.Div("✓ You have been signed out.", className='msg-ok')
        return None, SHOW, HIDE, msg

    # ── LOGIN
    if triggered == 'login-btn':
        if not username or not password:
            msg = html.Div("⚠ Please enter both username and password.", className='msg-error')
            return current_auth, SHOW, HIDE, msg

        try:
            test_shelter = AnimalShelter(username, password)
            result = test_shelter.read({})
            if result is None:
                raise Exception("Empty result — verify credentials.")

            auth_data = {
                'authenticated': True,
                'username':      username,
                'password':      password,
                'login_time':    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            }
            # Switch to dashboard — clear feedback
            return auth_data, HIDE, SHOW, ""

        except Exception as e:
            err = str(e)
            if 'Authentication failed' in err or 'auth failed' in err.lower():
                err = "Authentication failed — incorrect username or password."
            elif 'Connection refused' in err or 'timeout' in err.lower():
                err = "Could not reach the database server."
            msg = html.Div(f"⚠ {err}", className='msg-error')
            return None, SHOW, HIDE, msg

    return current_auth, SHOW, HIDE, ""


# ── THEME TOGGLE ──────────────────────────────────────────────────────────
@app.callback(
    [Output('root',        'data-theme'),
     Output('theme-store', 'data'),
     Output('theme-btn',   'children')],
    Input('theme-btn', 'n_clicks'),
    prevent_initial_call=False,
)
def toggle_theme(n):
    n = n or 0
    if n % 2 == 0:
        return 'dark',  'dark',  '☀ Light Mode'
    return     'light', 'light', '🌙 Dark Mode'


# ── TABLE THEMING ─────────────────────────────────────────────────────────
@app.callback(
    [Output('datatable-id', 'style_header'),
     Output('datatable-id', 'style_cell'),
     Output('datatable-id', 'style_data_conditional'),
     Output('datatable-id', 'style_filter')],
    Input('theme-store', 'data'),
)
def theme_table(theme):
    t = THEMES[theme]
    hdr = {
        'backgroundColor': t['surface2'], 'color': t['text'],
        'fontWeight': '700', 'fontSize': '.7rem',
        'letterSpacing': '.12em', 'textTransform': 'uppercase',
        'padding': '11px 15px',
        'borderBottom': f"2px solid {t['accent']}",
        'fontFamily': '"Syne", sans-serif',
    }
    cell = {
        'textAlign': 'left', 'padding': '9px 15px',
        'fontFamily': '"JetBrains Mono", monospace', 'fontSize': '.78rem',
        'border': 'none', 'backgroundColor': t['surface'],
        'color': t['text'],
        'minWidth': '130px', 'maxWidth': '220px',
        'whiteSpace': 'nowrap', 'overflow': 'hidden', 'textOverflow': 'ellipsis',
    }
    cond = [
        {'if': {'row_index': 'odd'},  'backgroundColor': t['row_odd']},
        {'if': {'row_index': 'even'}, 'backgroundColor': t['row_even']},
        {'if': {'state': 'selected'},
         'backgroundColor': 'rgba(88,166,255,.15)',
         'border': f"1px solid {t['accent']} !important"},
    ]
    filt = {'backgroundColor': t['surface2'], 'color': t['text'], 'border': 'none'}
    return hdr, cell, cond, filt


# ── FILTER PANEL COLLAPSE ─────────────────────────────────────────────────
@app.callback(
    Output('filter-panel-wrap', 'style'),
    Input('panel-btn', 'n_clicks'),
)
def toggle_panel(n):
    return {'display': 'none'} if (n or 0) % 2 == 1 else {}


# ── CLOCK ─────────────────────────────────────────────────────────────────
@app.callback(
    Output('live-clock', 'children'),
    Input('clock-tick', 'n_intervals'),
)
def update_clock(_):
    return datetime.now().strftime("%a %b %d  %H:%M")


# ── USER BADGE ────────────────────────────────────────────────────────────
@app.callback(
    Output('user-badge', 'children'),
    Input('auth-store', 'data'),
)
def update_badge(auth):
    if not auth:
        return ""
    return [html.Span("👤"), html.Span(auth.get('username', ''))]


# ── MAIN DATA CALLBACK ────────────────────────────────────────────────────
@app.callback(
    [Output('datatable-id',   'data'),
     Output('datatable-id',   'columns'),
     Output('filtered-store', 'data'),
     Output('rec-count',      'children'),
     Output('status-bar',     'children')],
    [Input('filter-type',  'value'),
     Input('age-slider',   'value'),
     Input('sex-filter',   'value'),
     Input('type-filter',  'value'),
     Input('name-search',  'value'),
     Input('auth-store',   'data')],
)
def update_data(filter_type, age_range, sex_vals, type_vals, name_query, auth):
    if not auth or not auth.get('authenticated'):
        return [], [], [], "", []

    shelter = AnimalShelter(auth['username'], auth['password'])

    query   = {}
    age_min, age_max = (age_range or [0, 520])

    if filter_type == 'water':
        query = {
            "breed": {"$in": ["Labrador Retriever Mix",
                               "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
        }
    elif filter_type == 'mountain':
        query = {
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute",
                               "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
        }
    elif filter_type == 'disaster':
        query = {
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd",
                               "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
        }

    data = shelter.read(query)
    df   = pd.DataFrame.from_records(data)

    if df.empty:
        return [], [], [], "0 records", [html.Span("No records match the current filters.")]

    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)

    if 'age_upon_outcome_in_weeks' in df.columns:
        df = df[(df['age_upon_outcome_in_weeks'] >= age_min) &
                (df['age_upon_outcome_in_weeks'] <= age_max)]

    if sex_vals and 'sex_upon_outcome' in df.columns:
        df = df[df['sex_upon_outcome'].isin(sex_vals)]

    if type_vals and 'animal_type' in df.columns:
        known = [v for v in type_vals if v != 'Other']
        if 'Other' in type_vals:
            df = df[df['animal_type'].isin(known) | ~df['animal_type'].isin(['Dog', 'Cat'])]
        else:
            df = df[df['animal_type'].isin(known)]

    if name_query and name_query.strip() and 'name' in df.columns:
        df = df[df['name'].str.contains(name_query.strip(), case=False, na=False)]

    if len(df) > 1000:
        df = df.head(1000)

    COLS = ['animal_id','name','breed','color','animal_type',
            'sex_upon_outcome','age_upon_outcome_in_weeks',
            'location_lat','location_long','outcome_type']
    display_cols = [c for c in COLS if c in df.columns]
    columns = [{"name": c.replace('_',' ').title(), "id": c,
                 "deletable": False, "selectable": True} for c in display_cols]

    ts = datetime.now().strftime("%H:%M:%S")
    status = [
        html.Span(f"📋  {len(df):,} records",        style={'marginRight':'20px'}),
        html.Span(f"🔎  {filter_type.upper()}",       style={'marginRight':'20px'}),
        html.Span(f"🕑  Age {age_min}–{age_max} wks", style={'marginRight':'20px'}),
        html.Span(f"⏱  Updated {ts}"),
    ]

    return (df[display_cols].to_dict('records'), columns,
            df.to_dict('records'), f"{len(df):,} records", status)


# ── KPI CARDS ─────────────────────────────────────────────────────────────
@app.callback(
    Output('kpi-row', 'children'),
    [Input('filtered-store', 'data'),
     Input('theme-store',    'data')],
)
def update_kpis(store, theme):
    t  = THEMES[theme]
    df = pd.DataFrame.from_records(store) if store else pd.DataFrame()
    n  = len(df)

    breeds  = df['breed'].nunique() if 'breed' in df.columns and n else 0
    avg_age = df['age_upon_outcome_in_weeks'].mean() \
              if 'age_upon_outcome_in_weeks' in df.columns and n else 0
    pct_int = (df['sex_upon_outcome'].str.contains('Intact', na=False).mean() * 100
               if 'sex_upon_outcome' in df.columns and n else 0)
    dogs    = int((df['animal_type'] == 'Dog').sum()) if 'animal_type' in df.columns and n else 0

    defs = [
        ("📋", f"{n:,}",             "Total Records",  t['accent']),
        ("🐾", f"{breeds}",          "Unique Breeds",  t['accent2']),
        ("📅", f"{avg_age:.1f} wks", "Avg Age",        t['accent3']),
        ("✅", f"{pct_int:.0f}%",    "Intact Animals", t['accent4']),
        ("🐕", f"{dogs:,}",         "Dogs",           "#e3b341"),
    ]

    return [
        html.Div(className='kpi-card', style={'--kpi-color': color}, children=[
            html.Div(icon, className='kpi-icon'),
            html.Div(val,  className='kpi-val', style={'color': color}),
            html.Div(lbl,  className='kpi-lbl'),
        ])
        for icon, val, lbl, color in defs
    ]


# ── CHART TYPE ────────────────────────────────────────────────────────────
@app.callback(
    Output('chart-type-store', 'data'),
    [Input('pie-btn', 'n_clicks'),
     Input('bar-btn', 'n_clicks')],
    prevent_initial_call=True,
)
def set_chart_type(pie_n, bar_n):
    triggered = callback_context.triggered[0]['prop_id'].split('.')[0]
    return 'bar' if triggered == 'bar-btn' else 'pie'


# ── BREED CHART ───────────────────────────────────────────────────────────
@app.callback(
    Output('breed-chart', 'figure'),
    [Input('datatable-id',    'derived_virtual_data'),
     Input('chart-type-store','data'),
     Input('theme-store',     'data')],
)
def breed_chart(view, chart_type, theme):
    t    = THEMES[theme]
    tmpl = t['plotly_tmpl']
    base = dict(paper_bgcolor=t['surface'], plot_bgcolor=t['surface'],
                font=dict(family='Syne', color=t['text']),
                margin=dict(l=12, r=12, t=16, b=10))

    if not view:
        return go.Figure().update_layout(template=tmpl, **base)

    dff = pd.DataFrame.from_dict(view)
    if 'breed' not in dff.columns:
        return go.Figure().update_layout(template=tmpl, **base)

    top = dff['breed'].value_counts().head(10).reset_index()
    top.columns = ['breed', 'count']

    if chart_type == 'bar':
        fig = px.bar(top, x='count', y='breed', orientation='h',
                     color='count',
                     color_continuous_scale=['#1f4068', t['accent']],
                     template=tmpl)
        fig.update_layout(**base, coloraxis_showscale=False,
                          yaxis={'categoryorder':'total ascending'},
                          xaxis_title='Count', yaxis_title='')
    else:
        fig = px.pie(top, values='count', names='breed',
                     color_discrete_sequence=px.colors.qualitative.Pastel,
                     template=tmpl)
        fig.update_layout(**base, legend=dict(font=dict(family='JetBrains Mono', size=10)))

    fig.update_layout(transition_duration=350)
    return fig


# ── SEX CHART ─────────────────────────────────────────────────────────────
@app.callback(
    Output('sex-chart', 'figure'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('theme-store',  'data')],
)
def sex_chart(view, theme):
    t    = THEMES[theme]
    tmpl = t['plotly_tmpl']
    base = dict(paper_bgcolor=t['surface'], plot_bgcolor=t['surface'],
                font=dict(family='Syne', color=t['text']),
                margin=dict(l=12, r=12, t=16, b=10))

    if not view:
        return go.Figure().update_layout(template=tmpl, **base)

    dff = pd.DataFrame.from_dict(view)
    if 'sex_upon_outcome' not in dff.columns:
        return go.Figure().update_layout(template=tmpl, **base)

    counts = dff['sex_upon_outcome'].value_counts().reset_index()
    counts.columns = ['sex', 'count']

    fig = px.bar(counts, x='sex', y='count', color='sex',
                 color_discrete_sequence=[t['accent'], t['accent2'],
                                          t['accent3'], t['accent4']],
                 template=tmpl)
    fig.update_layout(**base, showlegend=False, xaxis_title='', yaxis_title='Count',
                      transition_duration=350)
    return fig


# ── MAP ───────────────────────────────────────────────────────────────────
@app.callback(
    [Output('map-id',    'children'),
     Output('map-label', 'children')],
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows'),
     Input('theme-store',  'data')],
)
def update_map(view, sel_rows, theme):
    t = THEMES[theme]

    if not view:
        return (html.P("No data available.",
                       style={'padding':'30px','textAlign':'center','color':'var(--muted)'}), "")

    dff = pd.DataFrame.from_dict(view)
    row = sel_rows[0] if sel_rows else 0

    try:
        lat   = float(dff.iloc[row]['location_lat'])
        lon   = float(dff.iloc[row]['location_long'])
        breed = str(dff.iloc[row].get('breed', 'Unknown'))
        name  = str(dff.iloc[row].get('name',  'Unknown'))
        sex   = str(dff.iloc[row].get('sex_upon_outcome', ''))
        age   = dff.iloc[row].get('age_upon_outcome_in_weeks', '')
    except (KeyError, IndexError, ValueError):
        return (html.P("Select a row to see location.",
                       style={'padding':'30px','textAlign':'center','color':'var(--muted)'}), "")

    label = f"{name}  ·  {breed}  ·  ({lat:.4f}, {lon:.4f})"

    map_obj = dl.Map(
        style={'width': '100%', 'height': '400px'},
        center=[lat, lon], zoom=13,
        children=[
            dl.TileLayer(url=t['map_tile'],
                         attribution='© CartoDB  © OpenStreetMap contributors'),
            dl.Marker(position=[lat, lon], children=[
                dl.Tooltip(breed),
                dl.Popup([
                    html.Div(style={'fontFamily':'"Syne",sans-serif',
                                    'minWidth':'160px','padding':'4px'}, children=[
                        html.H3(name, style={'marginBottom':'6px','fontSize':'1rem'}),
                        html.P(breed, style={'fontFamily':'"JetBrains Mono",monospace',
                                             'fontSize':'.82rem','marginBottom':'4px'}),
                        html.P(sex,   style={'fontSize':'.78rem','color':'#777','marginBottom':'2px'}),
                        html.P(f"Age: {age} wks" if age != '' else "",
                               style={'fontSize':'.78rem','color':'#777'}),
                        html.P(f"{lat:.5f}, {lon:.5f}",
                               style={'fontFamily':'"JetBrains Mono",monospace',
                                      'fontSize':'.7rem','color':'#999','marginTop':'6px'}),
                    ]),
                ]),
            ]),
        ],
    )
    return map_obj, label


# ── CSV EXPORT ────────────────────────────────────────────────────────────
@app.callback(
    Output('download-csv', 'data'),
    Input('export-btn', 'n_clicks'),
    State('filtered-store', 'data'),
    prevent_initial_call=True,
)
def export_csv(_, store):
    if not store:
        return None
    df    = pd.DataFrame.from_records(store)
    fname = f"grazioso_salvare_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    return dcc.send_data_frame(df.to_csv, fname, index=False)


# =========================================================================
# 7. RUN
# =========================================================================
if __name__ == '__main__':
    port = 8060
    try:
        codio_domain = os.environ.get('CODIO_BOX_DOMAIN')
        print(f"\n▶  DASHBOARD → https://{codio_domain}-{port}.codio.io\n")
    except Exception:
        pass
    app.run_server(mode='external', host='0.0.0.0', port=port)


▶  DASHBOARD → https://None-8060.codio.io

Dash app running on http://0.0.0.0:8060/
